# Classification fédérée du diabète par FedFiFA++
### Fittest Forest Aggregation — sélection des meilleurs arbres et reconstruction d'un modèle global



**Structure :**
- Étape 0 — Chargement et découpage des données
- Étape 1 — Accès aux arbres individuels d'une forêt
- Étape 2 — Évaluation de la qualité de chaque arbre
- Étape 3 — Répartition entre les clients (IID / non-IID)
- Étape 4 — Sélection des meilleurs arbres (Top-M)
- Étape 5 — Reconstruction de la forêt globale (vote pondéré)
- Étape 6 — Rondes fédérées itératives
- Étape 7 — Application aux deux scénarios + référence centralisée
- Étape 8 — Résultats et figures

## Importation des bibliothèques et paramètres

In [ ]:
# Bibliothèques
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

# Paramètres de reproductibilité
RANDOM_STATE    = 42     # graine aléatoire
K               = 5      # nombre de clients (établissements)
N_TREES         = 60     # arbres par forêt locale
N_ROUNDS        = 5      # rondes de communication fédérées
F_UPLOAD        = 20     # meilleurs arbres envoyés par chaque client
M_SELECT        = 60     # arbres retenus par le serveur (taille du GFEM)
THRESHOLD       = 0.5    # seuil de décision
ALPHA_DIRICHLET = 0.5    # paramètre d'hétérogénéité (non-IID)

np.random.seed(RANDOM_STATE)

 Chargement et découpage des données

In [ ]:
# Charge le fichier CSV dans un tableau pandas (une ligne = un patient)
df = pd.read_csv('reforme_diabetes_dataset.csv')

# Sépare les variables explicatives (X) de la cible (y)
X = df.drop('diabetes', axis=1).values   # X : les 6 variables cliniques, sans la colonne cible
y = df['diabetes'].values                # y : la cible (0 = non-diabétique, 1 = diabétique)

# Réserve 20 % des patients pour le test final (jamais vus pendant l'entraînement)
# stratify=y conserve la proportion 60/40 dans les deux morceaux
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# Prélève un lot de validation DANS l'entraînement (servira à noter les arbres)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=RANDOM_STATE, stratify=y_train)

# Affiche la taille de chaque sous-ensemble pour vérification
print(f"Entraînement : {X_tr.shape[0]} patients")
print(f"Validation   : {X_val.shape[0]} patients")
print(f"Test         : {X_test.shape[0]} patients")
print(f"Répartition cible : {y.mean():.3f} diabétiques")


def evaluer(y_true, y_pred, y_proba):
    """Calcule les cinq métriques de classification à partir des vraies étiquettes,
    des prédictions de classe et des probabilités prédites."""
    return {
        'exactitude': accuracy_score(y_true, y_pred),    # proportion de bonnes prédictions
        'precision':  precision_score(y_true, y_pred),   # fiabilité des prédictions positives
        'rappel':     recall_score(y_true, y_pred),      # capacité à détecter les diabétiques
        'f1':         f1_score(y_true, y_pred),           # moyenne harmonique précision/rappel
        'auc':        roc_auc_score(y_true, y_proba),     # capacité de séparation des classes
    }

## Référence — Random Forest centralisé
Modèle entraîné sur toutes les données réunies : c'est le plafond de performance.

In [ ]:
# Crée une forêt de 100 arbres ; n_jobs=-1 utilise tous les cœurs du processeur
rf_central = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)

# Entraîne la forêt sur TOUTES les données d'entraînement (scénario idéal, sans confidentialité)
rf_central.fit(X_train, y_train)

# Prédit la probabilité d'être diabétique pour chaque patient de test ([:, 1] = colonne classe 1)
proba_c = rf_central.predict_proba(X_test)[:, 1]

# Transforme la probabilité en décision 0/1 en appliquant le seuil
pred_c  = (proba_c >= THRESHOLD).astype(int)

# Calcule les métriques du modèle centralisé (référence)
res_central = evaluer(y_test, pred_c, proba_c)

print("RF centralisé (référence)")
for k, v in res_central.items():
    print(f"  {k:11s}: {v:.4f}")

## Étape 1 — Accès aux arbres individuels d'une forêt

In [ ]:
# Entraîne une petite forêt de démonstration sur un échantillon de 5000 patients
demo = RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1)
demo.fit(X_tr[:5000], y_tr[:5000])

# .estimators_ donne accès à la LISTE des arbres individuels de la forêt
print(f"Nombre d'arbres dans la forêt : {len(demo.estimators_)}")

# On peut faire prédire UN SEUL arbre (le premier) sur 3 patients : c'est la base de FedFiFA++
proba_1arbre = demo.estimators_[0].predict_proba(X_val[:3])[:, 1]
print(f"Probabilités du 1er arbre (3 patients) : {proba_1arbre.round(3)}")

## Étape 2 — Évaluation de la qualité de chaque arbre

In [ ]:
def score_arbre(arbre, X_val, y_val):
    """Attribue un score de qualité Q à un arbre : précision - pénalité de complexité."""
    # Précision de l'arbre sur le lot de validation
    acc = accuracy_score(y_val, arbre.predict(X_val))
    # Nombre de feuilles de l'arbre (mesure de sa complexité)
    complexite = arbre.get_n_leaves()
    # Score = précision, diminuée d'une petite pénalité proportionnelle à la complexité
    return acc - 0.05 * (complexite / 10000.0)

# Calcule le score de tous les arbres de la forêt de démonstration
scores_demo = [score_arbre(a, X_val, y_val) for a in demo.estimators_]

# Affiche l'étendue des scores : ils varient d'un arbre à l'autre, d'où l'intérêt de sélectionner
print(f"Score min   : {min(scores_demo):.4f}")
print(f"Score max   : {max(scores_demo):.4f}")
print(f"Score moyen : {np.mean(scores_demo):.4f}")

## Étape 3 — Répartition entre les clients (IID / non-IID)

In [ ]:
def partition_iid(n, k):
    """Répartition HOMOGÈNE : mélange les indices puis les coupe en k parts égales."""
    # np.random.permutation(n) : mélange les numéros de 0 à n-1
    # np.array_split(..., k)   : coupe en k tranches de taille quasi égale
    return np.array_split(np.random.permutation(n), k)


def partition_non_iid(y, k, alpha):
    """Répartition HÉTÉROGÈNE : chaque classe est distribuée inégalement via Dirichlet."""
    # Crée k listes vides, une par client, qui recevront les indices des patients
    client_idx = [[] for _ in range(k)]

    # Traite chaque classe séparément (0 puis 1) : c'est ce qui crée le déséquilibre
    for classe in np.unique(y):
        # Récupère les positions des patients de la classe courante
        ci = np.where(y == classe)[0]
        # Mélange ces positions au hasard
        np.random.shuffle(ci)
        # Tire k proportions aléatoires qui somment à 1 (loi de Dirichlet)
        proportions = np.random.dirichlet([alpha] * k)
        # Transforme les proportions en points de coupure dans la liste des patients
        coupures = (np.cumsum(proportions) * len(ci)).astype(int)[:-1]
        # Découpe la liste aux points de coupure et distribue chaque tranche à un client
        for c, part in enumerate(np.split(ci, coupures)):
            client_idx[c].extend(part)

    # Convertit les listes en tableaux numpy et les renvoie
    return [np.array(c) for c in client_idx]

# Construit les deux partitions
parts_iid = partition_iid(len(X_tr), K)
parts_ni  = partition_non_iid(y_tr, K, ALPHA_DIRICHLET)

# Vérifie les tailles et l'hétérogénéité (proportion de diabétiques par client)
print("IID     — tailles :", [len(p) for p in parts_iid])
print("non-IID — tailles :", [len(p) for p in parts_ni])
print("non-IID — % diabétiques par client :", [f"{y_tr[p].mean():.2f}" for p in parts_ni])

## Étape 4 — Sélection des meilleurs arbres (Top-M)

In [ ]:
def meilleurs_arbres_client(rf, X_val, y_val, f=F_UPLOAD):
    """Retourne les f meilleurs arbres d'un client, chacun accompagné de son score."""
    # Associe à chaque arbre son score de qualité, sous forme de paires (arbre, score)
    scored = [(a, score_arbre(a, X_val, y_val)) for a in rf.estimators_]
    # Trie les paires par score décroissant (les meilleurs en tête)
    scored.sort(key=lambda t: t[1], reverse=True)
    # Ne garde que les f premiers (les meilleurs)
    return scored[:f]


def selection_globale(candidats, M=M_SELECT):
    """Le serveur retient les M meilleurs arbres parmi tous ceux reçus des clients."""
    # Trie tous les arbres candidats par score décroissant
    candidats.sort(key=lambda t: t[1], reverse=True)
    # Ne garde que les M meilleurs au niveau global
    return candidats[:M]

## Étape 5 — Reconstruction de la forêt globale (GFEM)

In [ ]:
class ForetGlobale:
    """Modèle global reconstruit : vote pondéré des meilleurs arbres sélectionnés."""

    def __init__(self, arbres_scores):
        # Sépare les arbres de leurs scores : ne garde que les arbres
        self.arbres = [a for a, s in arbres_scores]
        # Récupère les scores (max(s, 1e-6) évite une division par zéro plus bas)
        sc = np.array([max(s, 1e-6) for a, s in arbres_scores])
        # Transforme les scores en poids qui somment à 1 (les meilleurs arbres pèsent plus)
        self.poids = sc / sc.sum()

    def predict_proba(self, X):
        # Récupère la probabilité de classe 1 prédite par chaque arbre (une ligne par arbre)
        P = np.array([a.predict_proba(X)[:, 1] for a in self.arbres])
        # Moyenne pondérée à travers les arbres (axis=0), patient par patient : c'est le vote
        return np.average(P, axis=0, weights=self.poids)

    def predict(self, X, seuil=THRESHOLD):
        # Applique le seuil à la probabilité agrégée pour obtenir une décision 0/1
        return (self.predict_proba(X) >= seuil).astype(int)

## Étape 6 — Rondes fédérées itératives

In [ ]:
def fedfifa(X_tr, y_tr, X_val, y_val, X_test, y_test, partitions, label=""):
    """Procédure fédérée complète : rondes d'entraînement local, sélection et reconstruction."""
    foret_globale = None     # le modèle global (sera reconstruit à chaque ronde)
    pool_precedent = []      # mémoire : meilleurs arbres conservés d'une ronde à l'autre
    historique = []          # suivi du F1 par ronde (pour la courbe de convergence)

    # Boucle sur les rondes de communication
    for r in range(N_ROUNDS):
        candidats = []       # arbres candidats reçus des clients à cette ronde

        # --- Phase locale : chaque client entraîne sa forêt et envoie ses meilleurs arbres ---
        for p in partitions:
            # Entraîne une forêt UNIQUEMENT sur les indices p du client (localité des données)
            # random_state + r : produit de nouveaux arbres à chaque ronde
            rf = RandomForestClassifier(
                n_estimators=N_TREES, random_state=RANDOM_STATE + r,
                max_features='sqrt', bootstrap=True, n_jobs=-1)
            rf.fit(X_tr[p], y_tr[p])
            # Ajoute les meilleurs arbres de ce client à la liste des candidats
            candidats.extend(meilleurs_arbres_client(rf, X_val, y_val, F_UPLOAD))

        # --- Phase serveur : mémoire + sélection Top-M + reconstruction ---
        # Ajoute les meilleurs arbres conservés de la ronde précédente (mémoire itérative)
        candidats.extend(pool_precedent)
        # Sélectionne les M meilleurs arbres au global
        top = selection_globale(candidats, M_SELECT)
        # Conserve la moitié supérieure pour la ronde suivante
        pool_precedent = top[:M_SELECT // 2]
        # Reconstruit la forêt globale avec les arbres sélectionnés
        foret_globale = ForetGlobale(top)

        # Mesure le F1 de validation à cette ronde et l'enregistre
        f1_val = f1_score(y_val, foret_globale.predict(X_val))
        historique.append(f1_val)
        print(f"   [{label}] ronde {r+1}/{N_ROUNDS}  F1(validation) = {f1_val:.4f}")

    # --- Évaluation finale sur le test (jamais vu pendant l'entraînement) ---
    proba = foret_globale.predict_proba(X_test)
    pred  = (proba >= THRESHOLD).astype(int)
    return evaluer(y_test, pred, proba), historique, foret_globale

## Étape 7 — Application aux deux scénarios

In [ ]:
# Lance FedFiFA++ sur la répartition homogène (IID)
print("=== FedFiFA++ — scénario IID ===")
parts_iid = partition_iid(len(X_tr), K)
res_iid, hist_iid, gf_iid = fedfifa(X_tr, y_tr, X_val, y_val, X_test, y_test, parts_iid, "IID")

# Lance FedFiFA++ sur la répartition hétérogène (non-IID)
print("\n=== FedFiFA++ — scénario non-IID ===")
parts_ni = partition_non_iid(y_tr, K, ALPHA_DIRICHLET)
res_ni, hist_ni, gf_ni = fedfifa(X_tr, y_tr, X_val, y_val, X_test, y_test, parts_ni, "non-IID")

## Étape 8 — Résultats et figures

In [ ]:
# Rassemble les trois configurations dans un tableau récapitulatif
tableau = pd.DataFrame({
    'RF centralisé':     res_central,
    'FedFiFA++ IID':     res_iid,
    'FedFiFA++ non-IID': res_ni,
}).T.round(4)          # .T transpose (une ligne par approche), arrondi à 4 décimales
print(tableau)
tableau.to_csv('resultats_fedfifa.csv')   # sauvegarde en CSV

In [ ]:
# Figure 1 — Comparaison F1 / AUC des trois approches
TEAL, MINT = '#028090', '#02C39A'
modes = ['RF centralisé', 'FedFiFA++ IID', 'FedFiFA++ non-IID']
f1s  = [res_central['f1'],  res_iid['f1'],  res_ni['f1']]     # les trois F1
aucs = [res_central['auc'], res_iid['auc'], res_ni['auc']]    # les trois AUC
x = np.arange(len(modes)); w = 0.35                            # positions et largeur des barres
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, f1s, w, label='F1-score', color=TEAL)         # barres F1 à gauche
ax.bar(x + w/2, aucs, w, label='AUC', color=MINT)             # barres AUC à droite
for i, (a, b) in enumerate(zip(f1s, aucs)):                    # écrit les valeurs au-dessus
    ax.text(i - w/2, a + 0.005, f'{a:.3f}', ha='center', fontweight='bold', fontsize=9)
    ax.text(i + w/2, b + 0.005, f'{b:.3f}', ha='center', fontweight='bold', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(modes); ax.set_ylim(0.8, 1.02)
ax.set_ylabel('Score'); ax.set_title('Comparaison des performances : centralisé vs FedFiFA++')
ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig('fig_comparaison.png', dpi=130, bbox_inches='tight'); plt.show()

In [ ]:
# Figure 2 — Convergence du F1 au fil des rondes
AMBER = '#C9770A'
rounds = np.arange(1, N_ROUNDS + 1)                           # numéros de rondes (1 à 5)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rounds, hist_iid, 'o-', color=TEAL, lw=2, ms=8, label='IID')      # courbe IID
ax.plot(rounds, hist_ni, 's-', color=AMBER, lw=2, ms=8, label='non-IID')  # courbe non-IID
ax.set_xlabel('Ronde de communication'); ax.set_ylabel('F1-score (validation)')
ax.set_title('Convergence de FedFiFA++ au fil des rondes')
ax.set_xticks(rounds); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig('fig_convergence.png', dpi=130, bbox_inches='tight'); plt.show()

In [ ]:
# Figures 3-5 — Matrices de confusion des trois configurations
DEEP = '#024B57'
def plot_cm(y_true, y_pred, titre, fname):
    # Calcule les 4 quantités de la matrice de confusion
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    M = np.array([[tn, fp], [fn, tp]]); labels = [['VN','FP'],['FN','VP']]
    fig, ax = plt.subplots(figsize=(5, 4.2)); ax.imshow(M, cmap='Blues')
    # Écrit l'étiquette et l'effectif dans chaque case
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{labels[i][j]}\n{M[i,j]:,}', ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    color='white' if M[i,j] > M.max()/2 else DEEP)
    ax.set_xticks([0,1]); ax.set_xticklabels(['Prédit 0','Prédit 1'])
    ax.set_yticks([0,1]); ax.set_yticklabels(['Réel 0','Réel 1'])
    ax.set_title(titre); plt.tight_layout()
    plt.savefig(fname, dpi=130, bbox_inches='tight'); plt.show()

# Génère les trois matrices
plot_cm(y_test, pred_c,                  'RF centralisé',     'fig_cm_central.png')
plot_cm(y_test, gf_iid.predict(X_test),  'FedFiFA++ IID',     'fig_cm_iid.png')
plot_cm(y_test, gf_ni.predict(X_test),   'FedFiFA++ non-IID', 'fig_cm_noniid.png')

## Synthèse

L'écart entre le RF centralisé et FedFiFA++ mesure le **coût de la confidentialité**.
La courbe de convergence montre l'apport spécifique de FedFiFA++ : l'amélioration
progressive du modèle global au fil des rondes, particulièrement en non-IID. La
fédération est assurée par la sélection des arbres, la reconstruction du modèle global
et sa redistribution ; le vote pondéré demeure la règle de décision de la forêt
globale.